In [1]:
# === Cell 1: Setup ===
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
from google.colab import drive
drive.mount('/content/drive')

EVAL_PATH = "/content/drive/MyDrive/cs639_guardrails/data/eval_500.csv"
SAVE_DIR = "/content/drive/MyDrive/cs639_guardrails/s3_activations_eval"
import os; os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
# === Cell 2: Load + sanity check ===
eval_df = pd.read_csv(EVAL_PATH)
print(f"Rows: {len(eval_df)}")
print(f"Label distribution: {eval_df['off_topic'].value_counts().to_dict()}")


Rows: 500
Label distribution: {0: 250, 1: 250}


In [3]:
# === Cell 3: Load model ===
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
NUM_LAYERS = model.config.num_hidden_layers

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [4]:
# === Cell 4: Hooks (same fix as before) ===
activations = {}
def make_hook(layer_idx):
    def hook_fn(module, inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        if hidden.dim() != 3:
            raise ValueError(f"Layer {layer_idx}: shape {hidden.shape}")
        activations[layer_idx] = hidden[:, -1, :].detach().cpu().float()
    return hook_fn

hooks = []
for i, layer in enumerate(model.model.layers):
    hooks.append(layer.register_forward_hook(make_hook(i)))

In [5]:
# === Cell 5: Extract ===
all_acts = {i: [] for i in range(NUM_LAYERS)}
all_labels = []
all_indices = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    messages = [
        {"role": "system", "content": row["system_prompt"]},
        {"role": "user", "content": row["prompt"]}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    with torch.no_grad():
        model(**inputs)

    for layer_idx in range(NUM_LAYERS):
        all_acts[layer_idx].append(activations[layer_idx].numpy())
    all_labels.append(int(row["off_topic"]))
    all_indices.append(int(row["__index_level_0__"]))


100%|██████████| 500/500 [00:28<00:00, 17.31it/s]


In [6]:
# === Cell 6: Save ===
np.save(f"{SAVE_DIR}/labels.npy", np.array(all_labels))
np.save(f"{SAVE_DIR}/indices.npy", np.array(all_indices))
for layer_idx in range(NUM_LAYERS):
    stacked = np.vstack(all_acts[layer_idx])
    np.save(f"{SAVE_DIR}/layer_{layer_idx:02d}.npy", stacked)
print(f"Saved {NUM_LAYERS} layer files + labels to {SAVE_DIR}")

Saved 32 layer files + labels to /content/drive/MyDrive/cs639_guardrails/s3_activations_eval
